# Data Cleaning Notebook
Week 4 IDX Exchange


Jack Phelan

# Setup

In [43]:
#imports
import pandas as pd
import re

In [44]:
# read in data
enriched_listings = pd.read_csv("./../data/enriched/listings_with_rates.csv", low_memory=False)
enriched_sold = pd.read_csv("./../data/enriched/sold_with_rates.csv", low_memory=False)

# Datetime Conversions

In [45]:
def dt_converter(df): 
    # Convert date fields to datetime format
    date_fields = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
    for field in date_fields:
        if field in df.columns:
            df[field] = pd.to_datetime(df[field], errors='coerce')
    return df

In [46]:
enriched_listings.head(5)

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,BuyerAgencyCompensationType,BuyerAgencyCompensation,year_month
0,929000.0,1076194146,dianne@drector.com,NaN,NaN,Dianne,Rector,NaN,NaN,16882 Canyon Lane,...,Huntington Beach Union High,92649,NaN,330.0,1847.0,NaN,16882 Canyon Lane,NaN,NaN,2024-05
1,999999.0,1076194026,realestateby_denisegarcia@gmail.com,NaN,NaN,Denise,Garcia,NaN,NaN,8720 S 4th Avenue,...,Inglewood Unified,90305,NaN,0.0,8508.0,NaN,8720 S 4th Avenue,NaN,NaN,2024-05
2,1400000.0,1076193814,alizabethjames@hotmail.com,NaN,NaN,Alizabeth,James,33.858559,-116.542169,505 E Molino Road,...,Palm Springs Unified,92262,NaN,NaN,10890.0,NaN,505 E Molino Road,NaN,NaN,2024-05
3,4998888.0,1076193812,ernieramos62@yahoo.com,NaN,NaN,Ernesto,Ramos,NaN,NaN,3653 Halldale Avenue,...,NaN,90018,NaN,NaN,6192.0,NaN,3653 Halldale Avenue,NaN,NaN,2024-05
4,549000.0,1076193525,parsanina@yahoo.com,NaN,NaN,Nina,Parsa,NaN,NaN,1736 N Mcdivitt Avenue,...,Los Angeles Unified,90221,NaN,0.0,4113.0,NaN,1736 N Mcdivitt Avenue,NaN,NaN,2024-05


In [58]:
enriched_listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893594 entries, 0 to 893593
Data columns (total 85 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   OriginalListPrice             890150 non-null  float64       
 1   ListingKey                    893594 non-null  int64         
 2   ListAgentEmail                891222 non-null  object        
 3   CloseDate                     257985 non-null  datetime64[ns]
 4   ClosePrice                    233372 non-null  float64       
 5   ListAgentFirstName            888314 non-null  object        
 6   ListAgentLastName             893511 non-null  object        
 7   Latitude                      781122 non-null  float64       
 8   Longitude                     781848 non-null  float64       
 9   UnparsedAddress               891239 non-null  object        
 10  PropertyType                  893594 non-null  object        
 11  LivingArea   

## Task List
- Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate) OK
- Remove unnecessary or redundant columns
- Handle missing values appropriately
- Ensure numeric fields are properly typed
- Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0,
negative Bedrooms or Bathrooms

In [47]:
int_list = dt_converter(enriched_listings)

In [48]:
dt_cols = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
int_list[dt_cols].head(5)

,CloseDate,PurchaseContractDate,ListingContractDate,ContractStatusChangeDate
0,NaT,NaT,2024-05-10,2024-05-10
1,NaT,NaT,2024-05-30,2024-05-30
2,NaT,NaT,2024-05-21,2024-05-21
3,NaT,NaT,2024-05-29,2024-05-29
4,NaT,NaT,2024-05-27,2024-05-27


# Removing Unnecessary Columns

In [ ]:
cols = [
        "AboveGradeFinishedArea",
        'BelowGradeFinishedArea',
        "TaxAnnualAmount",
        "BuilderName",
        "ElementarySchoolDistrict",
        "BusinessType",
        "CoveredSpaces",
        "MiddleOrJuniorSchool",
        "MiddleOrJuniorSchoolDistrict",
    ]

In [50]:
int_list = int_list.drop(columns=cols,axis=1)

In [51]:
int_list.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893594 entries, 0 to 893593
Data columns (total 77 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            890150 non-null  float64       
 1   ListingKey                   893594 non-null  int64         
 2   ListAgentEmail               891222 non-null  object        
 3   CloseDate                    257985 non-null  datetime64[ns]
 4   ClosePrice                   233372 non-null  float64       
 5   ListAgentFirstName           888314 non-null  object        
 6   ListAgentLastName            893511 non-null  object        
 7   Latitude                     781122 non-null  float64       
 8   Longitude                    781848 non-null  float64       
 9   UnparsedAddress              891239 non-null  object        
 10  PropertyType                 893594 non-null  object        
 11  LivingArea                

In [52]:
agent_cols = [int for int in int_list.columns if re.search(r'Agent', int)]

agent_test = ['ListAgentFirstName', 'ListAgentLastName']
agent_test_1 = ['ListAgentFirstName.1', 'ListAgentLastName.1']
agent_test_full = agent_test + agent_test_1

first = ['ListAgentFirstName', 'ListAgentFirstName.1']
last = ['ListAgentLastName', 'ListAgentLastName.1']

In [53]:
int_list[agent_test_full].head(5)

,ListAgentFirstName,ListAgentLastName,ListAgentFirstName.1,ListAgentLastName.1
0,Dianne,Rector,Dianne,Rector
1,Denise,Garcia,Denise,Garcia
2,Alizabeth,James,Alizabeth,James
3,Ernesto,Ramos,Ernesto,Ramos
4,Nina,Parsa,Nina,Parsa


In [54]:
agent_drops = [
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFirstName.1",
    "ListAgentLastName.1",
]

In [56]:
int_list = int_list.drop(columns=agent_drops)

In [57]:
int_list.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893594 entries, 0 to 893593
Data columns (total 72 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            890150 non-null  float64       
 1   ListingKey                   893594 non-null  int64         
 2   CloseDate                    257985 non-null  datetime64[ns]
 3   ClosePrice                   233372 non-null  float64       
 4   Latitude                     781122 non-null  float64       
 5   Longitude                    781848 non-null  float64       
 6   UnparsedAddress              891239 non-null  object        
 7   PropertyType                 893594 non-null  object        
 8   LivingArea                   782759 non-null  float64       
 9   ListPrice                    891376 non-null  float64       
 10  DaysOnMarket                 893594 non-null  int64         
 11  ListOfficeName            

# Numeric Type Checking

In [ ]:
int_list = int_list.copy()

In [ ]:
parking_vals = int_list['ParkingTotal'].unique()

display(parking_vals) # are half parking spaces valid? idk

In [ ]:
to_int_cols = [
    "YearBuilt",
    "StreetNumberNumeric",
    "BathroomsTotalInteger",
    "TaxYear",
    "Stories",
]

int_list[to_int_cols] = int_list[to_int_cols].astype("Int64")